# Session 6 · Exercises

*Course team · Licence: CC BY 4.0*

The exercises for the 30 minutes of student work in each block. Work in pairs; one of you types, the other directs.
The lecturer chooses which exercises to set; there are more than fit into 30 minutes. Stretch tasks are for pairs that finish
early.

Formats: *Paper* (pen and paper or whiteboard, no laptop), *Predict* (write down the result before you run any
code), *Code* (laptop), *Pairs* (one types, one directs), *Class* (the whole room together).

Data: Inside Airbnb, Berlin snapshot of 26 June 2026 (CC BY 4.0), prepared once with `uv run python case-study/prepare_airbnb.py`. As in the lecture notebooks, prices are modelled for short-stay listings (minimum stay below 28 nights) only.

## Part 1 · The machine learning lifecycle and linear regression

Run the starter cell first.

In [ ]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
listings["km_to_centre"] = 111.2 * np.hypot(listings["latitude"] - 52.5219,                     # Alexanderplatz
                                            (listings["longitude"] - 13.4132) * np.cos(np.radians(52.52)))
short = listings[listings["price"].notna() & listings["minimum_nights"].lt(28)].copy()   # comparable prices
short["log_price"] = np.log(short["price"])
print(len(listings), "listings,", len(short), "short-stay listings with a price")

### 1.1 · A pricing aid along the lifecycle

*Pairs, 10 min*

> What would it take to turn a Berlin price model into a pricing aid for hosts? For each stage of the lifecycle, write one sentence on what it means for this task and which session covers it. Think about what is known when a host lists a *new* flat.

*Source:* original exercise for this course; background: the lifecycle in [01-lecture-lifecycle-and-linear-regression.ipynb](01-lecture-lifecycle-and-linear-regression.ipynb)

*Your answer:*

| Step | What it means for the price model | Session |
|---|---|---|
| 01 Problem definition | | |
| 02 Data collection | | |
| 03 Cleaning | | |
| 04 EDA | | |
| 05 Features | | |
| 06 Model selection | | |
| 07 Training | | |
| 08 Evaluation and tuning | | |
| 09 Deployment | | |
| 10 Monitoring | | |

### 1.2 · The same five lines, a model you have not seen

*Code, 10 min*

> Pick a regressor that the lectures did not use, for example `HuberRegressor` (`sklearn.linear_model`), `GradientBoostingRegressor` (`sklearn.ensemble`) or `SVR` (`sklearn.svm`), and look up its hyperparameters in its docstring. Split `X = short[["accommodates", "km_to_centre"]]` and `y = short["log_price"]` 80/20 with `random_state=42`, then run the same lines as in the lecture: construct, fit, predict three rows, score on the test data. Which learned attributes (ending in `_`) does your model have? Then wrap it in `make_pipeline(StandardScaler(), ...)`: does the test R² change, and why or why not?

*Source:* original exercise for this course; background: [02-lecture-scikit-learn-in-one-pattern.ipynb](02-lecture-scikit-learn-in-one-pattern.ipynb)

### 1.3 · Which line fails?

*Predict, 5 min*

> Write down for each line whether it runs or raises an error, and which error. Then run them, one per cell.
>
> 1. `LinearRegression().predict(X_test)`
> 2. `LinearRegression().fit(short["accommodates"], short["log_price"])`
> 3. `StandardScaler().fit_transform(short[["room_type"]])`
> 4. `LinearRegression().fit(short[["accommodates"]], short["log_price"]).coef_`

*Source:* original exercise for this course; background: [02-lecture-scikit-learn-in-one-pattern.ipynb](02-lecture-scikit-learn-in-one-pattern.ipynb)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

*Your answer:*

### 1.4 · Fit a line by hand

*Paper, 10 min*

> Points (1, 1), (2, 2), (3, 2).

*Source:* [Simple linear regression, Wikipedia](https://en.wikipedia.org/wiki/Simple_linear_regression)

*Your answer:*

### 1.5 · The coefficient of log price

*Paper, 5 min*

> A coefficient of +0.1 in a model of `log(price)` means?

*Source:* [UCLA OARC: interpreting models with log-transformed variables](https://stats.oarc.ucla.edu/other/mult-pkg/faq/general/faqhow-do-i-interpret-a-regression-model-when-some-variables-are-log-transformed/)

*Your answer:*

### 1.6 · Ten points, a polynomial of degree 9

*Predict, 5 min*

> What is the training R²? What will the test R² look like?

*Source:* [Bishop, *Pattern Recognition and Machine Learning* (2006), §1.1, free PDF](https://www.microsoft.com/en-us/research/uploads/prod/2006/01/Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf)

*Your answer:*

### 1.7 · A price model for private rooms

*Code, 15 min*

> Restrict `short` to private rooms. Split them 80/20 with `train_test_split(..., random_state=42)`, fit a linear regression of the log price on the number of guests, the distance to the centre and the district, and report the test MAE in euros against the median-price baseline of the training rooms. Interpret the coefficient of `accommodates` as a percentage. What does a separate model for private rooms allow that one model for all room types with a private-room dummy does not?

*Source:* original exercise for this course (Inside Airbnb Berlin case study)

## Part 2 · Other losses and other targets: robust and logistic regression

Run the starter cell of part 1 first (or this copy of it).

In [ ]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
listings["km_to_centre"] = 111.2 * np.hypot(listings["latitude"] - 52.5219,                     # Alexanderplatz
                                            (listings["longitude"] - 13.4132) * np.cos(np.radians(52.52)))
short = listings[listings["price"].notna() & listings["minimum_nights"].lt(28)].copy()   # comparable prices
short["log_price"] = np.log(short["price"])
print(len(listings), "listings,", len(short), "short-stay listings with a price")

### 2.1 · One number for everyone

*Paper, 5 min*

> You must predict one price for all listings. Which number minimises the MAE, and which the MSE?

*Source:* [Median, Wikipedia](https://en.wikipedia.org/wiki/Median)

*Your answer:*

### 2.2 · Log-odds in your head

*Paper, 5 min*

> A logistic coefficient of 0.69 means what for the odds?

*Source:* [UCLA OARC: odds ratios in logistic regression](https://stats.oarc.ucla.edu/other/mult-pkg/faq/general/faq-how-do-i-interpret-odds-ratios-in-logistic-regression/)

*Your answer:*

### 2.3 · Perfect separation

*Predict, then Code, 10 min*

> Fit a logistic regression where every customer older than 50 churned and no one younger did. What happens to the age coefficient?

*Source:* [Separation (statistics), Wikipedia](https://en.wikipedia.org/wiki/Separation_(statistics))

*Your answer:*

### 2.4 · How wide is the price range for a given size?

*Code, 15 min*

> For the entire homes in `short`, fit quantile regressions of the price in euros on `accommodates` for q = 0.1, 0.5 and 0.9, and least squares for comparison. Predict the three quantiles for a flat for two and for six guests. How does the range between the 10th and the 90th percentile change with size, and what would you tell a host with a flat for six guests?

*Source:* original exercise for this course (Inside Airbnb Berlin case study)

### 2.5 · Who shows a registration number?

*Code, 15 min*

> Take all entire homes and private rooms in `listings` (with or without a price). Fit a logistic regression of "the licence field holds a registration number" (`license_status == "registration number"`) on room type, a multi-listing host indicator (`calculated_host_listings_count > 1`) and `accommodates`. Report the odds ratios with their 95 % confidence intervals and interpret the coefficient of the multi-listing host indicator in one sentence. Why would adding the hotel rooms cause trouble (see exercise 2.3)?

*Source:* original exercise for this course (Inside Airbnb Berlin case study)

### 2.6 · Huber's ε (stretch)

*Code, 10 min, stretch task*

> Split `short` 80/20 with `random_state=42`. Fit `HuberRegressor` of the price in euros on `accommodates` and `km_to_centre` for ε = 1.1, 1.35, 2, 5 and 20, and least squares for comparison. How does the coefficient per km change as ε grows, and why?

*Source:* original exercise for this course (Inside Airbnb Berlin case study)